In [ ]:
%pip install flask

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

objetos = {}
proximo_id = 1

campos = ("descricao", "local", "data", "status")


def dados_validos(dados):
    return (
        isinstance(dados, dict)
        and all(
            isinstance(dados.get(campo), str)
            and dados[campo].strip()
            for campo in campos
        )
    )


# Listar todos os objetos
@app.get("/objetos")
def listar_objetos():
    return jsonify(list(objetos.values())), 200


# Consultar um objeto pelo ID
@app.get("/objeto")
def consultar_objeto():
    id_objeto = request.args.get("id", type=int)

    if id_objeto is None:
        return jsonify({"erro": "Informe um ID inteiro."}), 400

    if id_objeto not in objetos:
        return jsonify({"erro": "Objeto não encontrado."}), 404

    return jsonify(objetos[id_objeto]), 200


# Cadastrar um objeto
@app.post("/objeto")
def cadastrar_objeto():
    global proximo_id

    dados = request.get_json(silent=True)

    if not dados_validos(dados):
        return jsonify({
            "erro": "Preencha descricao, local, data e status com textos."
        }), 400

    objeto = {campo: dados[campo] for campo in campos}
    objeto["id"] = proximo_id

    objetos[proximo_id] = objeto
    proximo_id += 1

    return jsonify(objeto), 201


# Atualizar um objeto
@app.put("/objeto")
def atualizar_objeto():
    id_objeto = request.args.get("id", type=int)

    if id_objeto is None:
        return jsonify({"erro": "Informe um ID inteiro."}), 400

    if id_objeto not in objetos:
        return jsonify({"erro": "Objeto não encontrado."}), 404

    dados = request.get_json(silent=True)

    if not dados_validos(dados):
        return jsonify({
            "erro": "Preencha descricao, local, data e status com textos."
        }), 400

    for campo in campos:
        objetos[id_objeto][campo] = dados[campo]

    return jsonify(objetos[id_objeto]), 200


# Excluir um objeto
@app.delete("/objeto")
def excluir_objeto():
    id_objeto = request.args.get("id", type=int)

    if id_objeto is None:
        return jsonify({"erro": "Informe um ID inteiro."}), 400

    if id_objeto not in objetos:
        return jsonify({"erro": "Objeto não encontrado."}), 404

    del objetos[id_objeto]

    return "", 204

In [ ]:
cliente = app.test_client()


def mostrar(titulo, resposta):
    print(titulo)
    print("Código:", resposta.status_code)
    print("Resposta:", resposta.get_json(silent=True))
    print()


dados = {
    "descricao": "Estojo azul",
    "local": "Sala 203",
    "data": "2026-10-10",
    "status": "encontrado"
}

# 1. Cadastrar
resposta = cliente.post("/objeto", json=dados)
mostrar("CADASTRAR", resposta)

id_objeto = resposta.get_json()["id"]
rota = f"/objeto?id={id_objeto}"

# 2. Listar todos
mostrar("LISTAR TODOS", cliente.get("/objetos"))

# 3. Consultar pelo ID
mostrar("CONSULTAR UM", cliente.get(rota))

# 4. Atualizar
dados["status"] = "devolvido"
mostrar("ATUALIZAR", cliente.put(rota, json=dados))

# 5. Excluir
mostrar("EXCLUIR", cliente.delete(rota))

# 6. Verificar se foi excluído
mostrar("CONSULTAR APÓS EXCLUSÃO", cliente.get(rota))

# 7. Consultar sem informar o ID
mostrar(
    "CONSULTAR SEM ID",
    cliente.get("/objeto")
)

# 8. Consultar com um ID inválido
mostrar(
    "CONSULTAR COM ID INVÁLIDO",
    cliente.get("/objeto?id=abc")
)

# 9. Cadastrar com campos obrigatórios faltando
mostrar(
    "CADASTRAR COM CAMPOS FALTANDO",
    cliente.post("/objeto", json={"descricao": "Mochila"})
)

CADASTRAR
Código: 201
Resposta: {'data': '2026-10-10', 'descricao': 'Estojo azul', 'id': 1, 'local': 'Sala 203', 'status': 'encontrado'}

LISTAR TODOS
Código: 200
Resposta: [{'data': '2026-10-10', 'descricao': 'Estojo azul', 'id': 1, 'local': 'Sala 203', 'status': 'encontrado'}]

CONSULTAR UM
Código: 200
Resposta: {'data': '2026-10-10', 'descricao': 'Estojo azul', 'id': 1, 'local': 'Sala 203', 'status': 'encontrado'}

ATUALIZAR
Código: 200
Resposta: {'data': '2026-10-10', 'descricao': 'Estojo azul', 'id': 1, 'local': 'Sala 203', 'status': 'devolvido'}

EXCLUIR
Código: 204
Resposta: None

CONSULTAR APÓS EXCLUSÃO
Código: 404
Resposta: {'erro': 'Objeto não encontrado.'}

CONSULTAR SEM ID
Código: 400
Resposta: {'erro': 'Informe um ID inteiro.'}

CONSULTAR COM ID INVÁLIDO
Código: 400
Resposta: {'erro': 'Informe um ID inteiro.'}

CADASTRAR COM CAMPOS FALTANDO
Código: 400
Resposta: {'erro': 'Preencha descricao, local, data e status com textos.'}

